## LAB 01 — Ingestão Social (challenge 004)

Lê `dados/social_media_dataset.csv` e grava Delta. Colunas **somente as do CSV**. Não inventar `engagement_rate`.

### Controle de Alteração de Versões

| versão | data | autor | e-mail | alterações |
| --- | --- | --- | --- | --- |
| 1.2 | 21-SET-2026 | Adriano Aguiar | — | Troca para `social_media_dataset.csv` do challenge 004 |

### Contrato do CSV (contado no disco 2026-09-21)

| Arquivo | Linhas | Colunas |
| --- | ---: | --- |
| `social_media_dataset.csv` | 52214 | `id`, `platform`, `content_id`, `creator_id`, `creator_name`, `content_url`, `content_type`, `content_category`, `post_date`, `language`, `content_length`, `content_description`, `hashtags`, `views`, `likes`, `shares`, `comments_count`, `comments_text`, `follower_count`, `is_sponsored`, `disclosure_type`, `sponsor_name`, `sponsor_category`, `disclosure_location`, `audience_age_distribution`, `audience_gender_distribution`, `audience_location` |

Grain = post (`id` PK; `content_id` único e diferente de `id`). Sem dimensão — rollup = `GROUP BY`.

Armadilhas do dump (CSV ganha o brief):

- Brief ~52 000; disco = **52214 × 27**.
- **Não há** `engagement_rate`. Derivar: `(likes + shares + comments_count) / views` — no dump isso fica ≈ 0,199 ± 0,005.
- `creator_id` é string (`creator_586`), **não** int. `id` é o inteiro.
- `creator_name` **não** é PK (~10 nomes por `creator_id`). Use `creator_id`.
- `follower_count` varia por post, não é atributo estável do creator.
- Audiência é **bucket** (`19-25`, `male`, `Brazil`), não JSON de distribuição.
- `is_sponsored` no CSV = `TRUE`/`FALSE`. `disclosure_type` = `none` ⇔ orgânico.
- Sem coluna de custo: ROI de patrocínio é hipótese, não campo.
- Plataformas: Bilibili, YouTube, Instagram, RedNote, TikTok.
- `post_date` é string `M/D/YY h:mm AM` (ex. `5/29/23 12:15 AM`).

### Como executar no Databricks

1. Importe a pasta `notebook-marketing-004-social` para o Workspace (incluindo `dados/`).
2. Widgets: `catalog_name` (default `workspace`), `schema_name` (`social`), `data_path` (vazio = auto).
3. `data_path` se o auto-detect falhar: `/Workspace/Users/<voce>/notebook-marketing-004-social/dados`.
4. Cluster ou serverless → **Run All**.
5. Lab 02 e Lab 04 leem `workspace.social.social_media_dataset`.

In [ ]:
from pathlib import Path

EXPECTED = {
    "social_media_dataset": {
        "rows": 52214,
        "columns": [
            "id", "platform", "content_id", "creator_id", "creator_name",
            "content_url", "content_type", "content_category", "post_date",
            "language", "content_length", "content_description", "hashtags",
            "views", "likes", "shares", "comments_count", "comments_text",
            "follower_count", "is_sponsored", "disclosure_type", "sponsor_name",
            "sponsor_category", "disclosure_location", "audience_age_distribution",
            "audience_gender_distribution", "audience_location",
        ],
    },
}

INT_COLUMNS = {
    "id", "content_length", "views", "likes",
    "shares", "comments_count", "follower_count",
}
BOOL_COLUMNS = {"is_sponsored"}


def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def resolve_data_path(explicit: str) -> str:
    marker = "social_media_dataset.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-marketing-004-social" in parts:
            idx = parts.index("notebook-marketing-004-social")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
            candidates.append(Path("/") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend(
        [here / "dados", here.parent / "dados", here.parent.parent / "dados"]
    )
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError(
        "Não achei social_media_dataset.csv. Defina o widget data_path "
        "para a pasta dados/ no Workspace ou em um Volume."
    )


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "social")
data_path = resolve_data_path(widget_or_default("data_path", ""))
print(f"catalog={catalog_name} schema={schema_name}")
print(f"data_path={data_path}")

In [ ]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [ ]:
from pyspark.sql import functions as F

loaded = {}
for entity_name, contract in EXPECTED.items():
    csv_path = str(Path(data_path) / f"{entity_name}.csv")
    table_name = f"{catalog_name}.{schema_name}.{entity_name}"
    sdf = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("nullValue", "")
        .option("mode", "PERMISSIVE")
        .csv(csv_path)
    )
    missing = [col for col in contract["columns"] if col not in sdf.columns]
    extra = [col for col in sdf.columns if col not in contract["columns"]]
    if missing or extra:
        raise ValueError(
            f"{entity_name}: colunas fora do CSV. missing={missing} extra={extra}"
        )
    sdf = sdf.select(*contract["columns"])
    for column in sdf.columns:
        if column in INT_COLUMNS:
            sdf = sdf.withColumn(column, F.col(column).cast("int"))
        elif column in BOOL_COLUMNS:
            sdf = sdf.withColumn(column, F.col(column).cast("boolean"))
    row_count = sdf.count()
    if row_count != contract["rows"]:
        raise ValueError(
            f"{entity_name}: esperado {contract['rows']} linhas, lido {row_count}"
        )
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    loaded[entity_name] = {"rows": row_count, "cols": sdf.columns}
    print(f"{table_name} rows={row_count}")

loaded

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

In [ ]:
display(
    spark.table(f"{catalog_name}.{schema_name}.social_media_dataset").limit(5)
)